# Train BRIDGE models for the RBPs in different cell lines

This notebook documents the **training** workflow implemented in `main.py`.

Including:
- clear input/output expectations
- step-by-step visibility into each feature modality
- sanity checks like printing tensor shapes
- a faithful end-to-end training run driven by the original CLI

## 0. Environment & project root
We ensure the repo root is on `sys.path` so imports like `utils.BRIDGE` work.

In [1]:
import os, sys
from pathlib import Path

project_root = Path.cwd()
os.chdir(project_root.parent.parent)
project_root = Path.cwd()
print("project_root:", project_root)

assert (project_root / "utils").exists(), f"Cannot find utils/ under {project_root}. Open the notebook from repo root."
sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Python:", sys.version.split()[0])

project_root: /fs1/private/user/wangyubo/code/BRIDGE
Project root: /fs1/private/user/wangyubo/code/BRIDGE
Python: 3.10.10


## 1. Configure run parameters
These mirror the CLI arguments in `main.py`. Edit them as needed.

In [2]:
# --- Core inputs (match main.py defaults) ---
DATA_FILE = "AUH_HepG2"          # --data_file
DATA_PATH = "./dataset"          # --data_path
TRANSFORMER_PATH = "./RBPformer" # --Transformer_path

# --- Outputs ---
RESULTS_DIR = "./results"             # --results_dir
MODEL_SAVE_PATH = "./results/model"   # --model_save_path

# --- Runtime ---
SEED = 42
DEVICE_NUM = 0
USE_CPU = False   # True to force CPU

# --- Training hyperparams (CLI arguments) ---
LR = 0.001
EARLY_STOPPING = 10

# --- Constants from main.py ---
MAX_LENGTH = 101

# --- Sanity / smoke settings ---
SMOKE_TEST = True     # True: build embeddings/features for a small subset for quick shape checks
SMOKE_N = 8

print("DATA_FILE:", DATA_FILE)
print("DATA_PATH:", DATA_PATH)
print("TRANSFORMER_PATH:", TRANSFORMER_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("MODEL_SAVE_PATH:", MODEL_SAVE_PATH)

DATA_FILE: AUH_HepG2
DATA_PATH: ./dataset
TRANSFORMER_PATH: ./RBPformer
RESULTS_DIR: ./results
MODEL_SAVE_PATH: ./results/model


## 2. Check required input files
`main.py` expects two FASTA files:
- `{DATA_PATH}/{DATA_FILE}_neg.fa`
- `{DATA_PATH}/{DATA_FILE}_pos.fa`

In [3]:
from pathlib import Path

data_path = Path(DATA_PATH)
neg_fa = data_path / f"{DATA_FILE}_neg.fa"
pos_fa = data_path / f"{DATA_FILE}_pos.fa"

print("NEG FASTA:", neg_fa, "exists:", neg_fa.exists())
print("POS FASTA:", pos_fa, "exists:", pos_fa.exists())

assert neg_fa.exists() and pos_fa.exists(), (
    "Missing FASTA inputs. Please place *_neg.fa and *_pos.fa under DATA_PATH."
)

NEG FASTA: dataset/AUH_HepG2_neg.fa exists: True
POS FASTA: dataset/AUH_HepG2_pos.fa exists: True


## 3. Load sequences, structures, and labels
`read_fasta()` should return sequences + structures + binary labels.

In [4]:
from utils.dataloaders import read_fasta
import numpy as np

sequences, structs, labels = read_fasta(str(neg_fa), str(pos_fa))
sequences = list(sequences)
structs = list(structs)
labels = np.asarray(labels)

print("Num sequences:", len(sequences))
print("Num structs:", len(structs))
print("Labels shape:", labels.shape, "Pos ratio:", float(labels.mean()) if labels.size else None)

print("\nExample sequence:", sequences[0][:60] + ("..." if len(sequences[0]) > 60 else ""))
print("Example struct  :", structs[0][:60] + ("..." if len(structs[0]) > 60 else ""))

Num sequences: 15002
Num structs: 15002
Labels shape: (15002, 1) Pos ratio: 0.33328890800476074

Example sequence: GATGCTGTTGTGCATGGACAGCTCTCCAGTGGATTCGATGGGCCATAGCAATCCTGTGAT...
Example struct  : -1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,0.485,0.097,0.0...


## 4. Fix seeds and select device (same policy as `main.py`)

In [5]:
import random, torch, os
import numpy as np

def fix_seed(seed: int):
    torch.set_num_threads(1)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fix_seed(SEED)

device = torch.device("cpu") if USE_CPU else torch.device(f"cuda:{DEVICE_NUM}" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.set_device(DEVICE_NUM)

print("Device:", device)

Device: cuda:0


## 5. Build transformer embeddings and attention (RBPformer)
Calls `build_Transformer_embeddings(...)` and prints shapes.

In [7]:
from utils.gen_transformer_embedding import build_Transformer_embeddings

seq_for_run = sequences[:SMOKE_N] if SMOKE_TEST else sequences

Transformer_emb, attention_weight = build_Transformer_embeddings(
    sequences=list(seq_for_run),
    transformer_path=TRANSFORMER_PATH,
    device=device,
    k=1,
    transpose_to_ch_first=True,
)

def show_tensor(name, x):
    shp = getattr(x, "shape", None)
    dt = getattr(x, "dtype", None)
    dev = getattr(x, "device", None)
    print(f"{name:>18}: shape={shp} dtype={dt} device={dev} type={type(x)}")

show_tensor("Transformer_emb", Transformer_emb)
show_tensor("attention_weight", attention_weight)

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at ./RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Transformer_emb: shape=(8, 512, 101) dtype=float32 device=None type=<class 'numpy.ndarray'>
  attention_weight: shape=(8, 101, 103) dtype=float32 device=None type=<class 'numpy.ndarray'>


## 6. Build structure tensor (fixed length = 101)

In [8]:
from utils.structureFeatures import build_structure_tensor

struct_for_run = structs[:SMOKE_N] if SMOKE_TEST else structs
structure = build_structure_tensor(struct_for_run, MAX_LENGTH)

print("structure:", type(structure), "shape:", getattr(structure, "shape", None))

structure: <class 'numpy.ndarray'> shape: (8, 1, 101)


## 7. Load biochemical features (channel-first)

In [9]:
from utils.FeatureEncoding import dealwithdata

biochem = dealwithdata(DATA_FILE)
biochem = biochem.transpose([0, 2, 1])

try:
    biochem_for_run = biochem[:SMOKE_N] if SMOKE_TEST else biochem
except Exception:
    biochem_for_run = biochem

print("biochem:", type(biochem_for_run), "shape:", getattr(biochem_for_run, "shape", None))

[INFO] Encoded 15002 sequences for AUH_HepG2, shape: (15002, 101, 99)
biochem: <class 'numpy.ndarray'> shape: (8, 99, 101)


## 8. Load motif prior matrix

In [10]:
from utils.motif_prior.motif_prior import get_motif_prior_matrix

motif = get_motif_prior_matrix(DATA_FILE)
try:
    motif_for_run = motif[:SMOKE_N] if SMOKE_TEST else motif
except Exception:
    motif_for_run = motif

print("motif:", type(motif_for_run), "shape:", getattr(motif_for_run, "shape", None))

Valid motif file detected, skipping motif_prior: utils/motif_prior/output/AUH_HepG2/output/STRME_training_set.tab
motif: <class 'numpy.ndarray'> shape: (8, 1, 24)


## 9. Split modalities into train/test (aligned)

In [11]:
from utils.utils import split_dataset
labels_for_run = labels[:SMOKE_N] if SMOKE_TEST else labels

(train_emb, train_attn, train_struc, train_motif, train_biochem, train_label), \
(test_emb, test_attn, test_struc, test_motif, test_biochem, test_label) = split_dataset(
    Transformer_emb,
    attention_weight,
    structure,
    motif_for_run if SMOKE_TEST else motif,
    biochem_for_run if SMOKE_TEST else biochem,
    labels_for_run,
)

def show(name, x):
    print(f"{name:>10}: shape={getattr(x,'shape',None)} type={type(x)}")

print("=== Train split ===")
for n, x in [("emb", train_emb), ("attn", train_attn), ("struc", train_struc), ("motif", train_motif), ("biochem", train_biochem), ("label", train_label)]:
    show(n, x)

print("\n=== Test split ===")
for n, x in [("emb", test_emb), ("attn", test_attn), ("struc", test_struc), ("motif", test_motif), ("biochem", test_biochem), ("label", test_label)]:
    show(n, x)

=== Train split ===
       emb: shape=(7, 512, 101) type=<class 'numpy.ndarray'>
      attn: shape=(7, 101, 103) type=<class 'numpy.ndarray'>
     struc: shape=(7, 1, 101) type=<class 'numpy.ndarray'>
     motif: shape=(7, 1, 24) type=<class 'numpy.ndarray'>
   biochem: shape=(7, 99, 101) type=<class 'numpy.ndarray'>
     label: shape=(7, 1) type=<class 'numpy.ndarray'>

=== Test split ===
       emb: shape=(1, 512, 101) type=<class 'numpy.ndarray'>
      attn: shape=(1, 101, 103) type=<class 'numpy.ndarray'>
     struc: shape=(1, 1, 101) type=<class 'numpy.ndarray'>
     motif: shape=(1, 1, 24) type=<class 'numpy.ndarray'>
   biochem: shape=(1, 99, 101) type=<class 'numpy.ndarray'>
     label: shape=(1, 1) type=<class 'numpy.ndarray'>


## 10. Create Dataset & DataLoader (inspect one batch)

In [12]:
import torch
from torch.utils.data import DataLoader
from utils.utils import myDataset

train_set = myDataset(train_emb, train_attn, train_struc, train_motif, train_biochem, train_label)
test_set  = myDataset(test_emb,  test_attn,  test_struc,  test_motif,  test_biochem,  test_label)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=32*8, shuffle=False)

batch = next(iter(train_loader))
print("Batch contains", len(batch), "tensors (emb, attn, struc, motif, biochem, label).")
for i, t in enumerate(batch):
    print(f"batch[{i}] shape={getattr(t,'shape',None)} dtype={getattr(t,'dtype',None)}")

Batch contains 6 tensors (emb, attn, struc, motif, biochem, label).
batch[0] shape=torch.Size([7, 512, 101]) dtype=torch.float32
batch[1] shape=torch.Size([7, 101, 103]) dtype=torch.float32
batch[2] shape=torch.Size([7, 1, 101]) dtype=torch.float64
batch[3] shape=torch.Size([7, 1, 24]) dtype=torch.float64
batch[4] shape=torch.Size([7, 99, 101]) dtype=torch.float64
batch[5] shape=torch.Size([7, 1]) dtype=torch.float32


## 11. Initialize BRIDGE and (optional) sanity-check forward pass

In [13]:
from utils.BRIDGE import BRIDGE
from utils.utils import param_num

model = BRIDGE().to(device)
param_num(model)

emb, attn, struc, motif_, biochem_, y = batch
emb, attn, struc, motif_, biochem_ = emb.to(device), attn.to(device), struc.to(device), motif_.to(device), biochem_.to(device)

model.eval()
with torch.no_grad():
    try:
        out = model(emb, attn, struc, motif_, biochem_)
        print("Forward output shape:", getattr(out, "shape", None))
    except Exception as e:
        print("Forward call failed (signature mismatch or internal error).")
        print("Error:", e)
        print("Inputs:")
        print(" emb   :", emb.shape)
        print(" attn  :", attn.shape)
        print(" struc :", struc.shape)
        print(" motif :", motif_.shape)
        print(" biochem:", biochem_.shape)

---------------------------------
Total params: 19111841
Trainable params: 19111809
Non-trainable params: 32
---------------------------------
Forward call failed (signature mismatch or internal error).
Error: Input type (torch.cuda.DoubleTensor) and weight type (torch.cuda.FloatTensor) should be the same
Inputs:
 emb   : torch.Size([7, 512, 101])
 attn  : torch.Size([7, 101, 103])
 struc : torch.Size([7, 1, 101])
 motif : torch.Size([7, 1, 24])
 biochem: torch.Size([7, 99, 101])


## 12. Run full training (original CLI, no code modifications)

In [14]:
import subprocess, shlex

cmd = (
    f"python main.py "
    f"--train "
    f"--data_file {DATA_FILE} "
    f"--data_path {DATA_PATH} "
    f"--Transformer_path {TRANSFORMER_PATH} "
    f"--results_dir {RESULTS_DIR} "
    f"--model_save_path {MODEL_SAVE_PATH} "
    f"--seed {SEED} "
    f"--device_num {DEVICE_NUM} "
    f"--lr {LR} "
    f"--early_stopping {EARLY_STOPPING}"
)

if USE_CPU:
    cmd += " --use_cpu"

print("Command:\n", cmd)

subprocess.run(shlex.split(cmd), check=True)

Command:
 python main.py --train --data_file AUH_HepG2 --data_path ./dataset --Transformer_path ./RBPformer --results_dir ./results --model_save_path ./results/model --seed 42 --device_num 0 --lr 0.001 --early_stopping 10


/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at ./RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO] Encoded 15002 sequences for AUH_HepG2, shape: (15002, 101, 99)
Valid motif file detected, skipping motif_prior: utils/motif_prior/output/AUH_HepG2/output/STRME_training_set.tab
[RUN] AUH_HepG2_20260201-111327
[DIR] logs=results/logs model=results/model metrics=results/metrics
[CFG] results/logs/AUH_HepG2_20260201-111327_config.json
---------------------------------
Total params: 19111841
Trainable params: 19111809
Non-trainable params: 32
---------------------------------
AUH_HepG2 	 Train Epoch: 1     avg.loss: 0.9430 Acc: 0.67%, AUC: 0.7725, PRC: 0.6533, MCC: 0.3516, lr: 0.000040
AUH_HepG2 	 Test  Epoch: 1     avg.loss: 0.7414 Acc: 0.76%, AUC: 0.8128 (0.8128), PRC: 0.6915, MCC: 0.4406, 1


/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1179: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/fs1/private/user/wangyubo/code/BRIDGE/utils/metrics.py:188: RuntimeWarning: Mean of empty slice
  mean = [np.nanmean(correct), np.nanmean(auc_roc), np.nanmean(auc_pr), np.nanmean(f1), np.nanmean(mcc), tp, tn, fp, fn]
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


AUH_HepG2 	 Train Epoch: 2     avg.loss: 0.6604 Acc: 0.77%, AUC: nan, PRC: 0.7539, MCC: 0.5123, lr: 0.000080
AUH_HepG2 	 Test  Epoch: 2     avg.loss: 0.7006 Acc: 0.78%, AUC: 0.8403 (0.8403), PRC: 0.7282, MCC: 0.4827, 2
AUH_HepG2 	 Train Epoch: 3     avg.loss: 0.6287 Acc: 0.79%, AUC: 0.8651, PRC: 0.7775, MCC: 0.5461, lr: 0.000120
AUH_HepG2 	 Test  Epoch: 3     avg.loss: 0.7367 Acc: 0.78%, AUC: 0.8452 (0.8452), PRC: 0.7359, MCC: 0.4668, 3
AUH_HepG2 	 Train Epoch: 4     avg.loss: 0.6201 Acc: 0.79%, AUC: 0.8721, PRC: 0.7910, MCC: 0.5588, lr: 0.000160
AUH_HepG2 	 Test  Epoch: 4     avg.loss: 0.8686 Acc: 0.76%, AUC: 0.8478 (0.8478), PRC: 0.7413, MCC: 0.4178, 4
AUH_HepG2 	 Train Epoch: 5     avg.loss: 0.6048 Acc: 0.79%, AUC: 0.8800, PRC: 0.8007, MCC: 0.5705, lr: 0.000200
AUH_HepG2 	 Test  Epoch: 5     avg.loss: 0.6453 Acc: 0.79%, AUC: 0.8520 (0.8520), PRC: 0.7531, MCC: 0.5214, 5
AUH_HepG2 	 Train Epoch: 6     avg.loss: 0.6226 Acc: 0.78%, AUC: 0.8839, PRC: 0.8084, MCC: 0.5613, lr: 0.000240
AUH

...

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1179: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/fs1/private/user/wangyubo/code/BRIDGE/utils/metrics.py:188: RuntimeWarning: Mean of empty slice
  mean = [np.nanmean(correct), np.nanmean(auc_roc), np.nanmean(auc_pr), np

AUH_HepG2 	 Train Epoch: 33     avg.loss: 0.2800 Acc: 0.92%, AUC: nan, PRC: 0.9770, MCC: 0.8358, lr: 0.001320
AUH_HepG2 	 Test  Epoch: 33     avg.loss: 0.7782 Acc: 0.79%, AUC: 0.8679 (0.8827), PRC: 0.7774, MCC: 0.5402, 31
AUH_HepG2 	 Train Epoch: 34     avg.loss: 0.2914 Acc: 0.92%, AUC: 0.9887, PRC: 0.9789, MCC: 0.8404, lr: 0.001360
AUH_HepG2 	 Test  Epoch: 34     avg.loss: 1.3249 Acc: 0.80%, AUC: 0.8685 (0.8827), PRC: 0.7915, MCC: 0.5288, 31
AUH_HepG2 	 Train Epoch: 35     avg.loss: 0.2792 Acc: 0.93%, AUC: 0.9903, PRC: 0.9814, MCC: 0.8502, lr: 0.001400
AUH_HepG2 	 Test  Epoch: 35     avg.loss: 0.8355 Acc: 0.81%, AUC: 0.8646 (0.8827), PRC: 0.7859, MCC: 0.5649, 31
AUH_HepG2 	 Train Epoch: 36     avg.loss: 0.2640 Acc: 0.93%, AUC: 0.9894, PRC: 0.9793, MCC: 0.8539, lr: 0.001440
AUH_HepG2 	 Test  Epoch: 36     avg.loss: 2.9426 Acc: 0.72%, AUC: 0.8563 (0.8827), PRC: 0.7678, MCC: 0.3092, 31
AUH_HepG2 	 Train Epoch: 37     avg.loss: 0.2431 Acc: 0.93%, AUC: 0.9904, PRC: 0.9824, MCC: 0.8616, lr:

/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1179: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
/fs1/private/user/wangyubo/code/BRIDGE/utils/metrics.py:188: RuntimeWarning: Mean of empty slice
  mean = [np.nanmean(correct), np.nanmean(auc_roc), np.nanmean(auc_pr), np.nanmean(f1), np.nanmean(mcc), tp, tn, fp, fn]
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


AUH_HepG2 	 Train Epoch: 47     avg.loss: 0.1550 Acc: 0.97%, AUC: nan, PRC: 0.9954, MCC: 0.9312, lr: 0.001280
AUH_HepG2 	 Test  Epoch: 47     avg.loss: 1.9364 Acc: 0.80%, AUC: 0.8881 (0.8951), PRC: 0.8128, MCC: 0.5420, 44
AUH_HepG2 	 Train Epoch: 48     avg.loss: 0.0938 Acc: 0.98%, AUC: 0.9994, PRC: 0.9987, MCC: 0.9492, lr: 0.001280
AUH_HepG2 	 Test  Epoch: 48     avg.loss: 1.5102 Acc: 0.82%, AUC: 0.8849 (0.8951), PRC: 0.8113, MCC: 0.5836, 44
AUH_HepG2 	 Train Epoch: 49     avg.loss: 0.0921 Acc: 0.98%, AUC: 0.9989, PRC: 0.9980, MCC: 0.9462, lr: 0.001280
AUH_HepG2 	 Test  Epoch: 49     avg.loss: 1.4120 Acc: 0.83%, AUC: 0.8886 (0.8951), PRC: 0.8187, MCC: 0.5934, 44
AUH_HepG2 	 Train Epoch: 50     avg.loss: 0.0948 Acc: 0.98%, AUC: 0.9991, PRC: 0.9981, MCC: 0.9457, lr: 0.001024
AUH_HepG2 	 Test  Epoch: 50     avg.loss: 2.7685 Acc: 0.80%, AUC: 0.8851 (0.8951), PRC: 0.8170, MCC: 0.5313, 44
AUH_HepG2 	 Train Epoch: 51     avg.loss: 0.0716 Acc: 0.98%, AUC: 0.9996, PRC: 0.9990, MCC: 0.9554, lr:

CompletedProcess(args=['python', 'main.py', '--train', '--data_file', 'AUH_HepG2', '--data_path', './dataset', '--Transformer_path', './RBPformer', '--results_dir', './results', '--model_save_path', './results/model', '--seed', '42', '--device_num', '0', '--lr', '0.001', '--early_stopping', '10'], returncode=0)

## 13. Locate the latest run artifacts

In [15]:
from pathlib import Path
import json

metrics_dir = Path(RESULTS_DIR) / "metrics"
best_files = sorted(metrics_dir.glob(f"{DATA_FILE}_*_best.json"), key=lambda p: p.stat().st_mtime)

print("Found best summaries:", len(best_files))
if best_files:
    latest = best_files[-1]
    print("Latest best summary:", latest)
    with open(latest, "r", encoding="utf-8") as f:
        best = json.load(f)
    print(json.dumps(best, indent=2))
else:
    print("No best summary found yet. Train first.")

Found best summaries: 4
Latest best summary: results/metrics/AUH_HepG2_20260201-111327_best.json
{
  "run_name": "AUH_HepG2_20260201-111327",
  "data_file": "AUH_HepG2",
  "best_epoch": 44,
  "best_val_auc": 0.8950865000000001,
  "best_val_acc": 0.8383333333333334,
  "best_val_prc": 0.8295740030430263,
  "best_val_mcc": 0.630073216821487,
  "seed": 42,
  "device": "cuda:0",
  "checkpoint": "/fs1/private/user/wangyubo/code/BRIDGE/results/model/AUH_HepG2_20260201-111327.pth",
  "log_file": "/fs1/private/user/wangyubo/code/BRIDGE/results/logs/AUH_HepG2_20260201-111327.log",
  "config_file": "/fs1/private/user/wangyubo/code/BRIDGE/results/logs/AUH_HepG2_20260201-111327_config.json"
}
